# Step 1 — Data: loaders and frozen samples

**Goal:** turn RAGTruth, FaithBench and JudgeBench into one schema
(`id, task, source, subset, context, question, answer, answer_b, human_label, meta`) and freeze seeded samples.

| File | Rows | Purpose |
|---|---|---|
| `ragtruth_test_600.jsonl` | 600 = 3 task types × 2 labels × 100 | Task A main eval |
| `faithbench_250.jsonl` | 125 / 125 | Task A hard mode |
| `judgebench_eval_600.jsonl` | 600 | Task B eval |
| `judgebench_dev_20.jsonl` | 20 | Task B prompt debugging only |
| `ragtruth_pilot_50.jsonl` | 25 / 25 (train split) | Pilot (Step 6) |
| `ragtruth_fewshot_pool.jsonl` | 30 (train split) | M3 anchors are hand-picked from here |

Runtime: **CPU is enough** for this notebook.

In [ ]:
# --- 1. Setup: mount Drive, go to the project, install, import ---------------------------
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/LLM_as_Judge_Benchmark'   # <- change if you put it elsewhere
REPO_URL = ''   # optional: 'https://github.com/<you>/LLM_as_Judge_Benchmark.git' to clone instead

import os, sys, subprocess
if REPO_URL and not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
!pip -q install -r requirements.txt
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

import pandas as pd
pd.set_option('display.max_colwidth', 100); pd.set_option('display.width', 200)
print('cwd:', os.getcwd()); print(os.listdir('.'))

## 2. Unit tests first (offline, ~1 s)
These test the *logic* on tiny hand-made fixtures: label mapping, quality filter, length filter,
balanced sampling (exact cell counts, determinism, input-order invariance), JSONL round-trip.
The 5 `frozen` tests are skipped until `data/manifest.json` exists.

In [ ]:
!python -m pytest -q -rs tests/test_step1_data.py

## 3. Download raw data (pinned revisions) and look at it

In [ ]:
from ljbench import config, build_data as bd
print('RAGTruth  ', config.RAGTRUTH_REPO, config.RAGTRUTH_REVISION[:10])
print('JudgeBench', config.JUDGEBENCH_REPO, config.JUDGEBENCH_REVISION[:10])
print('FaithBench', config.FAITHBENCH_URL)

raw = bd.fetch_all_raw()
for name, df in raw.items():
    print(f'\n=== RAW {name}: {df.shape}')
    display(bd.preview(df, 2, 80))

**What to look for:** RAGTruth marks hallucinations as a JSON list of spans in `hallucination_labels`
(`'[]'` = clean). FaithBench gives per-summary `worst-label` / `best-label`. JudgeBench has `response_A`,
`response_B` and `label` = `A>B` / `B>A`.

## 4. Convert to the unified schema

In [ ]:
items = bd.convert_all(raw)          # also runs schema.validate() on each
for name, df in items.items():
    print(f'\n=== CONVERTED {name}: {len(df)} rows')
    display(bd.preview(df, 2, 70, ['id', 'subset', 'question', 'answer', 'human_label', 'meta']))
    display(bd.label_table(df))

**Explanation**
- RAGTruth test: 2,700 → **2,675** after dropping 25 `incorrect_refusal`/`truncated` rows.
  QA has only **160** hallucinated answers (the bottleneck for 100 per cell).
- FaithBench: 800 → **723** after dropping 77 `Questionable`. Only **238 SUPPORTED** (Consistent+Benign), so 125/125 is feasible.
- JudgeBench: 620 pairs, A wins 336 vs B 284 (slightly unbalanced; our A/B swap in Step 2 removes this bias).

## 5. Length profile (estimated tokens = chars / 3.5)

In [ ]:
import matplotlib.pyplot as plt
from ljbench.sampling import input_tokens
display(bd.length_profile(items))
fig, axes = plt.subplots(1, 4, figsize=(18, 3))
for ax, (name, df) in zip(axes, items.items()):
    input_tokens(df).hist(ax=ax, bins=40); ax.set_title(name); ax.axvline(config.MAX_INPUT_TOKENS, color='r', ls='--')
plt.tight_layout(); plt.show()

Budget: vLLM `max_model_len=8192` = input ≤ 6,000 + prompt template + ≤ 1,024 output tokens.
The longest item is ~3,600 estimated tokens, so **nothing is dropped**. Small models are not disadvantaged by truncation.

## 6. Sample and freeze

In [ ]:
from pathlib import Path
from ljbench.io_utils import sha256
samples, dropped = bd.build_samples(items)

# If frozen files already exist, DO NOT overwrite: rebuild elsewhere and prove they are identical.
target = Path('data') if not Path('data/manifest.json').exists() else Path('/content/rebuild')
manifest = bd.save(samples, target, dropped)
print('written to', target, '| dropped:', dropped)
if target != Path('data'):
    for f in manifest['files']:
        same = sha256(target / f) == sha256(Path('data') / f)
        print(('identical  ' if same else 'DIFFERENT! ') + f)
pd.DataFrame(manifest['files']).T[['rows', 'labels']]

## 7. Look at one frozen item from each file

In [ ]:
from ljbench.io_utils import read_jsonl
for f in sorted(Path('data').glob('*.jsonl')):
    df = read_jsonl(f)
    print(f'\n######## {f.name}  ({len(df)} rows)')
    display(bd.preview(df, 1, 160).T)
fb = read_jsonl('data/faithbench_250.jsonl'); jb = read_jsonl('data/judgebench_eval_600.jsonl')
display(pd.crosstab(fb.subset, fb.human_label)); display(pd.crosstab(jb.subset, jb.human_label))

## 8. Integration tests on the frozen files (gate)

In [ ]:
!python -m pytest -q -rs tests/test_step1_data.py

## Result and explanation

Expected: **17 passed, 0 skipped.**

| Check | What it proves |
|---|---|
| `test_hashes_match_manifest` | The frozen files have not been edited since they were built |
| `test_frozen_schema_and_sizes` | Every file has the right size and passes the schema validator |
| `test_frozen_balance` | RAGTruth has exactly 100 per (task type × label) cell; FaithBench is exactly 125/125 |
| `test_no_leakage_between_splits` | Pilot and few-shot items come from train, share no source document with test, and JudgeBench dev ∩ eval = ∅ |
| `test_all_within_length_budget` | No item can be truncated by an 8k context |

Notes for the blog / analysis:
- **Balanced sample ≠ real-world prevalence.** The balanced accuracy is fine as is, but precision must be reweighted to 10–20% (Step 3).
- **FaithBench generators are not balanced** (e.g. GPT-3.5 has 19 SUPPORTED / 7 UNSUPPORTED). Report FaithBench overall, not per generator.
- **JudgeBench eval is 323 A / 277 B.** Running both orders (Step 2) makes the effective gold set 50/50.
- The rebuild in cell 6 reproduces every file byte-for-byte, so the samples are fully determined by `config.py`.

**Gate passed → Step 2 (prompts + parsing).**